In [27]:
import pandas as pd
import numpy as np
from collections import defaultdict
from reactiva.config import DATASET_URI
from datetime import datetime
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


### Importando el dataset

In [3]:

df = pd.read_csv(DATASET_URI)

### procesamiento de datos pequeño para agregar session 

In [4]:
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'])
df['session'] = df['Purchase Date'].dt.month.apply(lambda x: 'winter' if x in(12,1,2) else 'summer' if  x in (3,4,5) else 'monsoon' if x in (6,7,8,9) else 'post-monsoon')
df_tovectorize = df[['Age','Gender','Location','session','Brand','Category','Online/Offline','Customer ID','Item Purchased','Purchase Date']]


df_purchases_270morethandays = df[df['Purchase Date']<=(df['Purchase Date'].max() - pd.Timedelta(days=270))]
df_purchases_270lessthandays = df[df['Purchase Date']>(df['Purchase Date'].max() - pd.Timedelta(days=270))]
cx_didnot_270daysago =np.setdiff1d(df_purchases_270morethandays['Customer ID'].unique(), df_purchases_270lessthandays['Customer ID'].unique())

# trayendo la temporada actual# 
month = pd.Timestamp.now().month

session = (
    'winter' if month in (12, 1, 2)
    else 'summer' if month in (3, 4, 5)
    else 'monsoon' if month in (6, 7, 8, 9)
    else 'post-monsoon')

session_list = pd.Series(['winter','summer','monsoon','post-monsoon'])
list_session = session_list[session_list != session]

### modelo de recomendación basado en usuari, pronóstico de compra basado en similitud de coseno de temporada anterior extrapolado and temporada actual, features de vector:
    Items Purchased
se crean los buckets basado en los usuarios port estaciones y sobre eso comparamos la similitud

In [8]:
def backtest_recommender_pr(df, holdout_session, session_list, k=5):
    train_sessions = session_list[session_list != holdout_session]
    df_train = df[df['session'].isin(train_sessions)]
    df_holdout_actual = df[df['session'] == holdout_session]

    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout_actual['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    session_matrices = {}
    for s in train_sessions:
        df_tovector = df_train[df_train['session'] == s]
        user_item_matrix = pd.crosstab(df_tovector['Customer ID'], df_tovector['Item Purchased'])
        similarity = cosine_similarity(user_item_matrix)
        session_matrices[s] = pd.DataFrame(
            similarity, index=user_item_matrix.index, columns=user_item_matrix.index
        )

    results = []
    for user in scoring_customers:
        user_dict = {}
        for s, similarity_df in session_matrices.items():
            if user not in similarity_df.columns:
                continue
            top5 = similarity_df[user].drop(user).sort_values(ascending=False).head(5)
            for sim_user, l in top5.items():
                if sim_user not in df_holdout_actual['Customer ID'].values:
                    continue
                user_dict[sim_user] = df_holdout_actual[df_holdout_actual['Customer ID'] == sim_user]

        # rank by frequency among neighbors instead of category-mode collapse,
        # so we actually have a ranked top-k list to evaluate
        if len(user_dict) > 1:
            df_users = pd.concat(user_dict.values(), ignore_index=True)
            recommendation = df_users['Item Purchased'].value_counts().head(k).index.tolist()
        else:
            recommendation = []

        actual = set(df_holdout_actual[df_holdout_actual['Customer ID'] == user]['Item Purchased'])
        precision, recall = precision_recall_at_k(recommendation, actual, k)

        results.append({
            'Customer ID': user,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall
        })

    results_df = pd.DataFrame(results)
    print(f'Precision@{k}: {results_df["precision@k"].mean():.3f}')
    print(f'Recall@{k}:    {results_df["recall@k"].mean():.3f}')
    return results_df

In [11]:
def precision_recall_at_k(recommended, actual, k):
    """
    recommended: list of recommended items (already ranked, best first)
    actual: set of items the customer actually bought
    k: cutoff
    """
    rec_k = recommended[:k]
    if not rec_k:
        return 0.0, 0.0
    hits = len(set(rec_k) & actual)
    precision = hits / len(rec_k)
    recall = hits / len(actual) if actual else 0.0
    return precision, recall

In [10]:
backtest_recommender_pr(df,'winter',session_list=session_list,k=5)

Precision@5: 0.072
Recall@5:    0.226


,Customer ID,recommended,actual,precision@k,recall@k
0,CUST000006,"[Kurta, Sneakers, Sandals, Shorts, Running Shoes]",[Wallet],0.0,0.0
1,CUST000008,"[Sneakers, Running Shoes, Trousers, T-shirt, F...",[Heels],0.0,0.0
2,CUST000011,"[Jacket, Running Shoes]",[Socks],0.0,0.0
3,CUST000014,"[T-shirt, Shorts, Sneakers, Saree, Running Shoes]","[Jacket, Running Shoes]",0.2,0.5
4,CUST000015,"[T-shirt, Shorts, Sneakers]",[Saree],0.0,0.0
...,...,...,...,...,...
1527,CUST003492,"[Formal Shoes, Dress, Jacket, Sunglasses]",[Kurta],0.0,0.0
1528,CUST003493,"[Running Shoes, Sandals, T-shirt, Formal Shoes...","[Sneakers, Slippers]",0.2,0.5
1529,CUST003495,"[Saree, Shorts, Shirt, Belt, Hoodie]",[Running Shoes],0.0,0.0
1530,CUST003497,"[Backpack, Sneakers, Shorts, Sandals, Wallet]","[Formal Shoes, Kurta]",0.0,0.0


### Creando un modelo de recomendación CF hybrid user y popularity
    Precisión@:
    Recall@:
    Hitting_rate:

In [12]:
def get_popularity_scores(df_train):
    """Item popularity across the full train set, normalized 0-1."""
    counts = df_train['Item Purchased'].value_counts()
    return counts / counts.max()

def get_cf_scores(user, session_matrices, df_holdout_actual):
    """Neighbor-frequency scores for a single user, normalized 0-1."""
    user_dict = {}
    for s, similarity_df in session_matrices.items():
        if user not in similarity_df.columns:
            continue
        top5 = similarity_df[user].drop(user).sort_values(ascending=False).head(5)
        for sim_user, l in top5.items():
            if sim_user not in df_holdout_actual['Customer ID'].values:
                continue
            user_dict[sim_user] = df_holdout_actual[df_holdout_actual['Customer ID'] == sim_user]

    if len(user_dict) < 1:
        return pd.Series(dtype=float)

    df_users = pd.concat(user_dict.values(), ignore_index=True)
    counts = df_users['Item Purchased'].value_counts()
    return counts / counts.max() if len(counts) else pd.Series(dtype=float)


def backtest_hybrid(df, holdout_session, session_list, alpha=0.3, k=5):
    """
    alpha: weight on CF score. (1-alpha) is weight on popularity.
    alpha=0 -> pure popularity, alpha=1 -> pure CF.
    """
    train_sessions = session_list[session_list != holdout_session]
    df_train = df[df['session'].isin(train_sessions)]
    df_holdout_actual = df[df['session'] == holdout_session]

    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout_actual['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    session_matrices = {}
    for s in train_sessions:
        df_tovector = df_train[df_train['session'] == s]
        user_item_matrix = pd.crosstab(df_tovector['Customer ID'], df_tovector['Item Purchased'])
        similarity = cosine_similarity(user_item_matrix)
        session_matrices[s] = pd.DataFrame(
            similarity, index=user_item_matrix.index, columns=user_item_matrix.index
        )

    pop_scores = get_popularity_scores(df_train)

    hits, total = 0, 0
    results = []
    for user in scoring_customers:
        cf_scores = get_cf_scores(user, session_matrices, df_holdout_actual)

        # combine on the union of items either method scored
        all_items = pop_scores.index.union(cf_scores.index)
        blended = pd.Series(0.0, index=all_items)
        blended = blended.add(alpha * cf_scores, fill_value=0)
        blended = blended.add((1 - alpha) * pop_scores, fill_value=0)

        recommendation = blended.sort_values(ascending=False).head(k).index.tolist()
        actual = set(df_holdout_actual[df_holdout_actual['Customer ID'] == user]['Item Purchased'])
        precision, recall = precision_recall_at_k(recommendation, actual, k)
    
        results.append({
            'Customer ID': user,
            'recommended': recommendation,
            'actual': list(actual),
            'precision@k': precision,
            'recall@k': recall
        })

        results_df = pd.DataFrame(results)
       
        hit = bool(set(recommendation) & actual)

        total += 1
        hits += hit
        results.append({'Customer ID': user, 'recommended': recommendation, 'actual': list(actual), 'hit': hit})
    print(f'Precision@{k}: {results_df["precision@k"].mean():.3f}')
    print(f'Recall@{k}:    {results_df["recall@k"].mean():.3f}')
    
    hit_rate = hits / total if total else None
    print(f'alpha={alpha}: hit rate = {hit_rate:.3f} ({hits}/{total})')
    return pd.DataFrame(results), hit_rate ,results_df

In [14]:
##trying the model above
for alpha in [0.0, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]:
    _, hr,results = backtest_hybrid(df_tovectorize, holdout_session='summer', session_list=session_list, alpha=alpha)

Precision@5: 0.112
Recall@5:    0.412
alpha=0.0: hit rate = 0.497 (795/1599)
Precision@5: 0.111
Recall@5:    0.410
alpha=0.1: hit rate = 0.495 (792/1599)
Precision@5: 0.111
Recall@5:    0.410
alpha=0.2: hit rate = 0.495 (792/1599)
Precision@5: 0.110
Recall@5:    0.405
alpha=0.3: hit rate = 0.490 (783/1599)
Precision@5: 0.097
Recall@5:    0.356
alpha=0.5: hit rate = 0.436 (697/1599)
Precision@5: 0.089
Recall@5:    0.323
alpha=0.7: hit rate = 0.405 (648/1599)
Precision@5: 0.067
Recall@5:    0.238
alpha=1.0: hit rate = 0.311 (498/1599)


### Creando un modelo de recomendación CF basado en items
    Precisión:
    Recall:

El modelo crea la matrix  de relación de items y despúes recomienda los items que aparecen con mayor frecuencia cuando un item es comprado

In [22]:

# ============================================================
# 1. DATA
# ============================================================

df_tovectorize = df[
    [
        'Customer ID',
        'Item Purchased'
    ]
].copy()

df_tovectorize = df_tovectorize.dropna(
    subset=['Customer ID', 'Item Purchased']
)


# ============================================================
# 2. ALL ITEMS
# ============================================================

all_items = sorted(
    df_tovectorize['Item Purchased'].unique()
)

id_map = {
    item: i
    for i, item in enumerate(all_items)
}


# ============================================================
# 3. BUILD ITEM-ITEM CO-OCCURRENCE MATRIX
# ============================================================

def build_item_item_matrix(df_train):

    co_count = defaultdict(
        lambda: defaultdict(int)
    )

    # Each customer contributes the items they purchased
    customer_items = (
        df_train
        .groupby('Customer ID')['Item Purchased']
        .apply(set)
    )

    for items in customer_items:

        items = list(items)

        for i in range(len(items)):

            for j in range(i + 1, len(items)):

                item_a = items[i]
                item_b = items[j]

                co_count[item_a][item_b] += 1
                co_count[item_b][item_a] += 1


    # Create item x item matrix
    matrix = np.zeros(
        (len(all_items), len(all_items)),
        dtype=float
    )

    for item, neighbors in co_count.items():

        i = id_map[item]

        for neighbor, count in neighbors.items():

            j = id_map[neighbor]

            matrix[i, j] = count


    return matrix


# ============================================================
# 4. ITEM-ITEM COSINE SIMILARITY
# ============================================================

def build_item_item_similarity(df_train):

    matrix = build_item_item_matrix(
        df_train
    )

    similarity = cosine_similarity(
        matrix
    )

    # Never recommend the trigger itself
    np.fill_diagonal(
        similarity,
        0
    )

    return similarity


# ============================================================
# 5. RECOMMENDATION FUNCTION
# ============================================================

def get_recommendations(
    trigger_item,
    similarity,
    top_n=5
):

    if trigger_item not in id_map:
        return []

    trigger_index = id_map[
        trigger_item
    ]

    scores = similarity[
        trigger_index
    ]

    # Rank highest similarity first
    ranked_indices = np.argsort(
        scores
    )[::-1]

    recommendations = []

    for index in ranked_indices:

        if scores[index] <= 0:
            continue

        item = all_items[index]

        if item == trigger_item:
            continue

        recommendations.append(item)

        if len(recommendations) >= top_n:
            break

    return recommendations


# ============================================================
# 6. PRECISION@K
# ============================================================

def precision_at_k(
    recommendations,
    actual_item,
    k
):

    recommendations = recommendations[:k]

    if len(recommendations) == 0:
        return 0.0

    return int(
        actual_item in recommendations
    ) / len(recommendations)


# ============================================================
# 7. RECALL@K
# ============================================================

def recall_at_k(
    recommendations,
    actual_item,
    k
):

    recommendations = recommendations[:k]

    # One held-out actual item
    if actual_item in recommendations:
        return 1.0

    return 0.0


# ============================================================
# 8. HIT RATE@K
# ============================================================

def hit_rate_at_k(
    recommendations,
    actual_item,
    k
):

    return int(
        actual_item in recommendations[:k]
    )


# ============================================================
# 9. EVALUATION
# ============================================================

def evaluate_item_item_cf(
    df_data,
    k=5
):

    results = []

    # --------------------------------------------------------
    # Evaluate each customer
    # --------------------------------------------------------

    for customer_id, customer_df in df_data.groupby(
        'Customer ID'
    ):

        customer_items = list(
            customer_df['Item Purchased'].unique()
        )

        # Need at least 2 different purchases
        if len(customer_items) < 2:
            continue


        # ----------------------------------------------------
        # Hold out the LAST item in the customer's list
        #
        # This is the actual / ground-truth item.
        #
        # NOTE:
        # Without dates, this is NOT temporal.
        # It is simply a leave-one-out evaluation.
        # ----------------------------------------------------

        actual_item = customer_items[-1]

        remaining_items = customer_items[:-1]


        # Need a trigger
        if len(remaining_items) == 0:
            continue

        trigger_item = remaining_items[-1]


        # ----------------------------------------------------
        # Build training data
        #
        # Start with ALL customers
        # ----------------------------------------------------

        df_train = df_data.copy()


        # Remove the customer's held-out item
        # so the model cannot directly learn that
        # customer's answer.
        mask = ~(
            (df_train['Customer ID'] == customer_id)
            &
            (df_train['Item Purchased'] == actual_item)
        )

        df_train = df_train[mask]


        # ----------------------------------------------------
        # Build ONE global item-item model
        # ----------------------------------------------------

        similarity = build_item_item_similarity(
            df_train
        )


        # ----------------------------------------------------
        # Generate recommendation from trigger
        # ----------------------------------------------------

        recommendations = get_recommendations(
            trigger_item,
            similarity,
            top_n=k
        )


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        precision = precision_at_k(
            recommendations,
            actual_item,
            k
        )

        recall = recall_at_k(
            recommendations,
            actual_item,
            k
        )

        hit_rate = hit_rate_at_k(
            recommendations,
            actual_item,
            k
        )


        results.append({

            'Customer ID':
                customer_id,

            'Trigger':
                trigger_item,

            'Actual':
                actual_item,

            'Recommendations':
                recommendations,

            f'Precision@{k}':
                precision,

            f'Recall@{k}':
                recall,

            f'HitRate@{k}':
                hit_rate

        })


    return pd.DataFrame(results)


# ============================================================
# 10. RUN
# ============================================================

results = evaluate_item_item_cf(
    df_tovectorize,
    k=5
)


# ============================================================
# 11. DISPLAY RESULTS
# ============================================================

display(
    results.head(20)
)


# ============================================================
# 12. OVERALL PERFORMANCE
# ============================================================

print("=" * 60)
print("ITEM-ITEM COLLABORATIVE FILTERING")
print("=" * 60)

print(
    f"Number of evaluation cases: {len(results):,}"
)

print(
    f"Precision@5: {results['Precision@5'].mean():.4f}"
)

print(
    f"Recall@5:    {results['Recall@5'].mean():.4f}"
)

print(
    f"HitRate@5:   {results['HitRate@5'].mean():.4f}"
)

,Customer ID,Trigger,Actual,Recommendations,Precision@5,Recall@5,HitRate@5
0,CUST000001,Heels,Bag,"[Backpack, Sunglasses, Boots, Hoodie, Socks]",0.0,0.0,0
1,CUST000002,Saree,Running Shoes,"[Sunglasses, Socks, Formal Shoes, Bag, Cap]",0.0,0.0,0
2,CUST000005,Shorts,T-shirt,"[Heels, Socks, Bag, Backpack, Shirt]",0.0,0.0,0
3,CUST000006,Shirt,Wallet,"[Dress, Jeans, Wallet, Sandals, Socks]",0.2,1.0,1
4,CUST000008,Running Shoes,Sandals,"[Bag, Jeans, Shirt, Cap, Socks]",0.0,0.0,0
5,CUST000011,Running Shoes,Socks,"[Bag, Jeans, Shirt, Socks, Cap]",0.2,1.0,1
6,CUST000012,Kurta,Wallet,"[Dress, Shirt, Sandals, Wallet, Jeans]",0.2,1.0,1
7,CUST000014,Slippers,T-shirt,"[Jeans, Belt, Wallet, Socks, Formal Shoes]",0.0,0.0,0
8,CUST000015,Saree,Jacket,"[Sunglasses, Socks, Formal Shoes, Bag, Cap]",0.0,0.0,0
9,CUST000016,Cap,Kurta,"[Boots, Socks, Shirt, Wallet, Sunglasses]",0.0,0.0,0


ITEM-ITEM COLLABORATIVE FILTERING
Number of evaluation cases: 2,651
Precision@5: 0.0273
Recall@5:    0.1366
HitRate@5:   0.1366


### Modelo de recomendación modelo de clasificación  Gradient Boosting
    Precision@:
    Recall@:
    Hitting Rate:

In [40]:


def build_customer_features(df_train):
    cat_counts = df_train.pivot_table(index='Customer ID', columns='Category', values='Item Purchased', aggfunc='count', fill_value=0)
    cat_counts.columns = [f'cat_count_{c}' for c in cat_counts.columns]
    agg = df_train.groupby('Customer ID').agg(total_purchases=('Item Purchased', 'count'), last_purchase=('Purchase Date', 'max'))
    reference_date = df_train['Purchase Date'].max()
    agg['days_since_last_purchase'] = (reference_date - agg['last_purchase']).dt.days
    agg = agg.drop(columns='last_purchase')
    return cat_counts.join(agg, how='inner')


def backtest_gradient_boosting(df, holdout_session, session_list, k=5, test_size=0.3, random_state=42):

    train_sessions = session_list[session_list != holdout_session]
    df_train = df[df['session'].isin(train_sessions)]
    df_holdout_actual = df[df['session'] == holdout_session]

    train_customers = df_train['Customer ID'].unique()
    holdout_customers = df_holdout_actual['Customer ID'].unique()
    scoring_customers = np.intersect1d(train_customers, holdout_customers)

    features = build_customer_features(df_train)
    features = features.loc[features.index.isin(scoring_customers)]

    labels = (
        df_holdout_actual[df_holdout_actual['Customer ID'].isin(features.index)]
        .groupby('Customer ID')['Category']
        .agg(lambda x: x.mode().iloc[0])
    )

    X = features
    y = labels.loc[X.index]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    clf = GradientBoostingClassifier(
        random_state=random_state
    )

    clf.fit(X_train, y_train)

    pred_category = pd.Series(
        clf.predict(X_test),
        index=X_test.index
    )

    item_pop_by_cat = (
        df_train.groupby(['Category', 'Item Purchased'])
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    precision_scores = []
    recall_scores = []
    hit_scores = []
    recommendation_results = []

    for user in X_test.index:

        cat = pred_category[user]

        top_items = (
            item_pop_by_cat[
                item_pop_by_cat['Category'] == cat
            ]['Item Purchased']
            .head(k)
            .tolist()
        )

        actual = set(
            df_holdout_actual[
                df_holdout_actual['Customer ID'] == user
            ]['Item Purchased']
        )

        hits = len(
            set(top_items) & actual
        )

        precision = (
            hits / len(top_items)
            if len(top_items) > 0
            else 0
        )

        recall = (
            hits / len(actual)
            if len(actual) > 0
            else 0
        )

        hit = int(hits > 0)

        precision_scores.append(precision)
        recall_scores.append(recall)
        hit_scores.append(hit)

        recommendation_results.append({
            'Customer ID': user,
            'Predicted Category': cat,
            'Recommendations': top_items,
            'Actual Items': list(actual),
            'Hits': hits,
            f'Precision@{k}': precision,
            f'Recall@{k}': recall,
            f'HitRate@{k}': hit
        })

    precision_at_k = np.mean(precision_scores)
    recall_at_k = np.mean(recall_scores)
    hit_rate = np.mean(hit_scores)

    results = pd.DataFrame(
        recommendation_results
    )

    print(f'Gradient Boosting results @ {k}')
    print(f'Precision@{k}: {precision_at_k:.3f}')
    print(f'Recall@{k}:    {recall_at_k:.3f}')
    print(f'Hit Rate@{k}:  {hit_rate:.3f} ({sum(hit_scores)}/{len(hit_scores)})')

    return clf, hit_rate, precision_at_k, recall_at_k, results




In [41]:
# ============================================================
# RUN MODEL
# ============================================================

session_list = np.array([
    'winter',
    'summer',
    'monsoon',
    'post-monsoon'
])

holdout_session = 'post-monsoon'

clf, hit_rate, precision, recall, results = backtest_gradient_boosting(
    df=df,
    holdout_session=holdout_session,
    session_list=session_list,
    k=5,
    test_size=0.3,
    random_state=42
)

display(results.head(20))

Gradient Boosting results @ 5
Precision@5: 0.091
Recall@5:    0.370
Hit Rate@5:  0.422 (151/358)


,Customer ID,Predicted Category,Recommendations,Actual Items,Hits,Precision@5,Recall@5,HitRate@5
0,CUST002084,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]","[Running Shoes, Jeans]",0,0.0,0.000000,0
1,CUST000608,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Kurta],1,0.2,1.000000,1
2,CUST000406,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Socks],0,0.0,0.000000,0
3,CUST002430,Accessories,"[Backpack, Socks, Bag, Belt, Sunglasses]",[Running Shoes],0,0.0,0.000000,0
4,CUST002416,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Kurta],1,0.2,1.000000,1
5,CUST000766,Footwear,"[Running Shoes, Sneakers, Sandals, Heels, Boots]",[Running Shoes],1,0.2,1.000000,1
6,CUST001945,Footwear,"[Running Shoes, Sneakers, Sandals, Heels, Boots]","[Backpack, Shorts]",0,0.0,0.000000,0
7,CUST001243,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Hoodie],0,0.0,0.000000,0
8,CUST001199,Footwear,"[Running Shoes, Sneakers, Sandals, Heels, Boots]",[Sunglasses],0,0.0,0.000000,0
9,CUST003189,Clothing,"[Kurta, Saree, Jacket, T-shirt, Shorts]",[Jacket],1,0.2,1.000000,1
